In [54]:
import torch 
import pandas as pd
import numpy as np 
import torch.nn as nn 
from datasets import load_dataset
from datasets import Dataset as HFDataset
from torch.utils.data import Dataset,DataLoader 
from tokenizers import Tokenizer,models,trainers,pre_tokenizers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [55]:
# Load Dataset from Hugginface libarary 
dataset = load_dataset('cfilt/iitb-english-hindi',split='train[:300000]')
dataset.features

{'translation': {'en': Value('string'), 'hi': Value('string')}}

In [56]:
# Delete the null values from dataset
dataset = dataset.filter(lambda x: 
                         x['translation']['en'] is not None and 
                         x['translation']['hi'] is not None and 
                         len(x['translation']['en'].strip())> 0 and
                         len(x['translation']['hi'].strip())> 0  
                        )

# Delete the duplicate values from dataset
df = dataset.flatten().to_pandas()
df = df.drop_duplicates(subset=['translation.en', 'translation.hi'],keep='first')
df = df.rename(columns={'translation.en':'en','translation.hi':'hi'})


In [57]:
MAX_CHAR_LEN = 200

# Rmove the row that is more than define size 
df = df[ 
        (df["en"].str.len() <= MAX_CHAR_LEN) &
        (df['hi'].str.len() <= MAX_CHAR_LEN)
       ].reset_index(drop=True)

# Quick distribution check
print(f"Avg English length (chars): {df['en'].str.len().mean():.1f}")
print(f"Avg Hindi length (chars):   {df['hi'].str.len().mean():.1f}")
print(f"Max English length (chars): {df['en'].str.len().max()}")
print(f"Max Hindi length (chars):   {df['hi'].str.len().max()}")

Avg English length (chars): 42.2
Avg Hindi length (chars):   46.9
Max English length (chars): 200
Max Hindi length (chars):   200


In [58]:
VOCAB_SIZE = 10000

#---For english vocab---
# Train joint BPE tokenizer
en_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
en_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Define special tokens for NMT
en_trainer = trainers.BpeTrainer(
    vocab_size = VOCAB_SIZE,
    special_tokens = ["[PAD]","[UNK]","[SOS]","[EOS]"]
)

en_tokenizer.train_from_iterator(df['en'].tolist(),trainer=en_trainer)

# ---For hindi vocab--- 
hi_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
hi_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

hi_trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[PAD]","[UNK]","[SOS]","[EOS]"]
)

hi_tokenizer.train_from_iterator(df['hi'].tolist(),trainer=hi_trainer)

print(f"English vocab size: {en_tokenizer.get_vocab_size()}")
print(f"Hindi vocab size:   {hi_tokenizer.get_vocab_size()}")







English vocab size: 10000
Hindi vocab size:   10000


In [59]:
# Saving vocab for refetch quickliy
en_tokenizer.save("/kaggle/working/en_tokenizer.json")
hi_tokenizer.save("/kaggle/working/hi_tokenizer.json")

In [60]:
MAX_TOKEN_LEN=50

# Get the token lenth of each sentense from vocab
def get_token_len(text,tokenizer):
    return len(tokenizer.encode(text).ids)

# Removing sentense more than define size
df['en_len'] = df['en'].apply(lambda x: get_token_len(x,en_tokenizer))
df['hi_len'] = df['hi'].apply(lambda x: get_token_len(x,hi_tokenizer))

df = df[
        (df['en_len'] <= MAX_TOKEN_LEN) &
        (df['hi_len'] <= MAX_TOKEN_LEN)
        ].reset_index(drop=True)

# Quick look on final dataset
print(f"Final dataset size after token filtering: {len(df)} pairs")
print(f"Avg English tokens: {df['en_len'].mean():.1f}")
print(f"Avg Hindi tokens:   {df['hi_len'].mean():.1f}")

Final dataset size after token filtering: 152845 pairs
Avg English tokens: 9.9
Avg Hindi tokens:   12.0


In [61]:
# Splitint dataset to train,validation and test 
from sklearn.model_selection import train_test_split

train_val_df,test_df = train_test_split(df,test_size=0.1,random_state=42)
train_df,val_df = train_test_split(train_val_df,test_size=0.111, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size:      {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size:       {len(test_df)}")

Train size:      122290
Validation size: 15270
Test size:       15285


In [62]:
# Dataset class 
class TranslationDataset(Dataset):
    
    def __init__(self,df,en_tokenizer,hi_tokenizer):
        self.df = df
        self.en_tokenizer = en_tokenizer
        self.hi_tokenizer = hi_tokenizer

        # defining special tokens
        self.en_pad = en_tokenizer.token_to_id("[PAD]")
        self.en_sos = en_tokenizer.token_to_id("[SOS]")
        self.en_eos = en_tokenizer.token_to_id("[EOS]")
        self.hi_pad = hi_tokenizer.token_to_id("[PAD]")
        self.hi_sos = hi_tokenizer.token_to_id("[SOS]")
        self.hi_eos = hi_tokenizer.token_to_id("[EOS]")
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self,index):
        
        en_text = self.df.loc[index,'en']
        hi_text = self.df.loc[index,'hi']

        # encode the df text to index number as in tokenizer
        en_ids = self.en_tokenizer.encode(en_text).ids
        hi_ids = self.hi_tokenizer.encode(hi_text).ids

        # added SOS and EOS on each line 
        src = [self.en_sos] + en_ids + [self.en_eos]
        trg = [self.hi_sos] + hi_ids + [self.hi_eos]

        # converting to tensor 
        src_tensor = torch.tensor(src,dtype= torch.long)
        trg_tensor = torch.tensor(trg,dtype= torch.long)

        return src_tensor,trg_tensor

In [63]:
from torch.nn.utils.rnn import pad_sequence

# adding padding according to the current batch
def collate_fn(batch):
    src,trg = zip(*batch)
    src_padded = pad_sequence(src,batch_first=True,padding_value=0)
    trg_padded = pad_sequence(trg,batch_first=True,padding_value=0)
    src_mask   = (src_padded != 0)
    return src_padded,trg_padded,src_mask

In [64]:
BATCH_SIZE=64

# Making Datasets 
train_dataset = TranslationDataset(train_df,en_tokenizer,hi_tokenizer)
val_dataset = TranslationDataset(val_df,en_tokenizer,hi_tokenizer)
test_dataset = TranslationDataset(test_df,en_tokenizer,hi_tokenizer)

# Making DataLoaders 
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_fn,num_workers=2,pin_memory=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2,pin_memory=True)

print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Train batches:      1911
Validation batches: 239
Test batches:       239


In [71]:
class Encoder(nn.Module):
    
    def __init__(self,en_vocab_size,embed_dim=256,hidden_size=512,num_layers=2):
        super().__init__()
        
        self.embedding = nn.Embedding(en_vocab_size,
                                      embed_dim,
                                      padding_idx=0)
        
        self.lstm = nn.LSTM(embed_dim,
                            hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=0.3)
        
        self.fc_hidden = nn.Linear(hidden_size*2,hidden_size)
        self.fc_cell = nn.Linear(hidden_size*2,hidden_size)
        
    def forward(self,src,src_mask=None):
        embedded = self.embedding(src)
        
        output,(hidden,cell) = self.lstm(embedded)
        
        hidden = torch.cat([hidden[-2],hidden[-1]],dim=1)
        hidden = torch.tanh(self.fc_hidden(hidden))
        hidden = hidden.unsqueeze(0)

        cell = torch.cat([cell[-2],cell[-1]],dim=1)
        cell = torch.tanh(self.fc_cell(cell))
        cell = cell.unsqueeze(0)

        return output,hidden,cell

In [72]:
class LuongAttention(nn.Module):
    
    def __init__(self,hidden_size):
        super().__init__()
        self.Wa = nn.Linear(hidden_size*2,hidden_size,bias=False)

    def forward(self,decoder_hidden,encoder_outputs):
        encoder_projected = self.Wa(encoder_outputs)
        # (4,10,512)
        decoder_hidden_t =decoder_hidden.permute(0,2,1)
        # (4,512,1)
        score = torch.bmm(encoder_projected,decoder_hidden_t)
        # (4,10,512)X(4,512,1)=(4,10,1)
        attention_weights = torch.softmax(score,dim=1)
        # (4,10,1) but sum of probability of each word in each sentence would be 1 
        attention_weights_t = attention_weights.permute(0,2,1)
        # (4,1,10)
        contex_vector = torch.bmm(attention_weights_t,encoder_outputs)
        # (4,1,10)X(4,10,1024) = (4,1,1024)
        return contex_vector,attention_weights


In [73]:
# Quick shape test 
HIDDEN_SIZE = 512
BATCH_SIZE  = 4
SRC_LEN     = 10

dummy_decoder_hidden   = torch.randn(BATCH_SIZE, 1, HIDDEN_SIZE)
dummy_encoder_outputs  = torch.randn(BATCH_SIZE, SRC_LEN, HIDDEN_SIZE * 2)

attention = LuongAttention(HIDDEN_SIZE)
context, weights = attention(dummy_decoder_hidden, dummy_encoder_outputs)

print(f"Context vector shape:    {context.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Weights sum to 1: {weights.squeeze(-1).sum(dim=1)}")

Context vector shape:    torch.Size([4, 1, 1024])
Attention weights shape: torch.Size([4, 10, 1])
Weights sum to 1: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [74]:
class Decoder(nn.Module):
    
    def __init__(self,hi_vocab_size,embed_dim=256,hidden_size=512,num_layers=1,dropout=0.3):
        super().__init__()

        self.hidden_size=hidden_size
        self.hi_vocab_size=hi_vocab_size
        
        self.embedding = nn.Embedding(hi_vocab_size,embed_dim,padding_idx=0)
        self.lstm = nn.LSTM(embed_dim + hidden_size * 2,
                            hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=False,
                            dropout = dropout if num_layers > 1 else 0,
                           )
        self.attention = LuongAttention(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(hidden_size + hidden_size*2,hi_vocab_size)

    def forward(self,tgt_token,decoder_hidden,decoder_cell,encoder_outputs):
            
        tgt_token = tgt_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt_token))
                
        decoder_hidden_for_att = decoder_hidden.permute(1,0,2)
        contex_vector,attention_weights = self.attention(decoder_hidden_for_att,encoder_outputs)
            
        lstm_input = torch.cat([embedded,contex_vector],dim=2)
        lstm_output,(decoder_hidden,decoder_cell) = self.lstm(lstm_input,(decoder_hidden,decoder_cell))
            
        lstm_output = lstm_output.squeeze(1)
        contex_vector = contex_vector.squeeze(1)
        combine = torch.cat([lstm_output,contex_vector],dim=1)
            
        prediction = self.fc_out(combine)
            
        return prediction,decoder_hidden,decoder_cell,attention_weights
            
        

In [79]:
class Seq2Seq(nn.Module):
    
    def __init__(self,encoder,decoder,device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device 

    def forward(self,src,trg,teacher_forcing_ratio=0.5):
        
        batch_size = src.shape[0]
        trg_seq_len = trg.shape[1]
        hi_vocab_size = self.decoder.hi_vocab_size

        all_prediction = torch.zeros(trg_seq_len,batch_size,hi_vocab_size).to(self.device)

        encoder_outputs,hidden,cell = self.encoder(src)
        decoder_input =trg[:,0]

        for t in range(1,trg_seq_len):
            prediction,hidden,cell,attention_weight = self.decoder(decoder_input,
                                                                    hidden,
                                                                    cell,
                                                                    encoder_outputs
                                                                   )
            all_prediction[t] = prediction
            use_tech_forsing = torch.rand(1).item() < teacher_forcing_ratio

            if use_tech_forsing:
                decoder_input = trg[:,t]
            else:
                decoder_input = prediction.argmax(dim=1)

        return all_prediction



In [80]:
# ── Hyperparameters ───────────────────────────────────────────────────
EN_VOCAB_SIZE = en_tokenizer.get_vocab_size()   # 10000
HI_VOCAB_SIZE = hi_tokenizer.get_vocab_size()   # 10000
EMBED_DIM     = 256
HIDDEN_SIZE   = 512
ENC_LAYERS    = 2
DEC_LAYERS    = 1
DROPOUT       = 0.3

# ── Build model ───────────────────────────────────────────────────────
encoder = Encoder(EN_VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, ENC_LAYERS).to(device)
decoder = Decoder(HI_VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, DEC_LAYERS, DROPOUT).to(device)
model   = Seq2Seq(encoder, decoder, device).to(device)

# ── Parameter count ───────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
# Expect roughly 25-35 million parameters

# ── Sanity check with one real batch ─────────────────────────────────
src_batch, trg_batch, src_mask = next(iter(train_loader))
src_batch = src_batch.to(device)
trg_batch = trg_batch.to(device)

output = model(src_batch, trg_batch, teacher_forcing_ratio=0.5)

print(f"Input src shape:    {src_batch.shape}")
# Expected: (64, src_seq_len)

print(f"Input trg shape:    {trg_batch.shape}")
# Expected: (64, trg_seq_len)

print(f"Model output shape: {output.shape}")
# Expected: (trg_seq_len, 64, 10000)

print("Seq2Seq wrapper working correctly")

Total trainable parameters: 35,191,568
Input src shape:    torch.Size([64, 44])
Input trg shape:    torch.Size([64, 51])
Model output shape: torch.Size([51, 64, 10000])
Seq2Seq wrapper working correctly
